# 01 - Data Setup: Medical Cost Personal Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/budisatrio32/Medical-Cost-Personal-Datasets/blob/main/notebooks/01_data_setup.ipynb)

Alur: **load → validasi → cleaning → fitur inovasi → split 80:20 → simpan**.

Notebook ini bisa dijalankan di **lokal** maupun **Google Colab**. Sel pertama otomatis mendeteksi Colab, meng-clone repo, dan meng-install library.

## 0. Environment (lokal / Colab)

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/budisatrio32/Medical-Cost-Personal-Datasets.git"
REPO_NAME = "Medical-Cost-Personal-Datasets"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content") / REPO_NAME
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    # Lokal: notebook ada di folder notebooks/, root project satu tingkat di atas
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Colab:", IN_COLAB, "| Root:", ROOT)

In [ ]:
import importlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config as cfg
import data_setup as ds
importlib.reload(cfg); importlib.reload(ds)

pd.set_option("display.max_columns", None)

## 1. Load data

In [ ]:
df_raw = ds.load_raw()
df_raw.head()

In [ ]:
df_raw.info()
df_raw.describe(include="all").T

## 2. Validasi

In [ ]:
ds.validate(df_raw)

## 3. Cleaning

In [ ]:
df_clean = ds.clean(df_raw)
df_clean.shape

## 4. Fitur inovasi

- `obese`: BMI ≥ 30
- `smoker_obese`: perokok **dan** obese
- `bmi_category`: kategori BMI WHO
- `age_group`: kelompok umur
- `log_charges`: log(1 + charges) untuk mengurangi skewness

In [ ]:
df = ds.add_features(df_clean)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["charges"], kde=True, ax=axes[0]).set_title("charges")
sns.histplot(df["log_charges"], kde=True, ax=axes[1]).set_title("log_charges")
plt.tight_layout(); plt.show()

## 5. Split train/test (80:20, stratify smoker) & simpan

In [ ]:
train, test = ds.split(df)
ds.save(df, train, test)

## 6. Cek helper untuk tahap modeling

In [ ]:
for fs in cfg.FEATURE_SETS:
    X_train, y_train = ds.get_xy(train, fs, log_target=True)
    print(f"{fs:<11} X_train: {X_train.shape} | fitur: {list(X_train.columns)}")